# Enterprise Pilot Data Readiness

## TL;DR

This notebook performs a privacy-safe, aggregate-only audit of the enterprise pilot dataset. It does not display raw conversations, email addresses, phone numbers, or ticket contents.

Current promotion rule: no candidate becomes active until a named business/knowledge owner approves it and its source evidence passes lifecycle checks.

## Context & methods

The audit checks tenant isolation, knowledge-source lifecycle, owner-reviewed golden coverage, behavior coverage, support outcome depth, and whether a redacted review queue exists. The minimum production target is 80 owner-reviewed rows; smoke rows are reported separately and do not count as production evidence.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from scripts.pilot_data_readiness import audit_readiness

DB_PATH = ROOT / 'data' / 'metadata.db'
KB_ID = 1
REVIEW_QUEUE = ROOT / 'data' / 'pilot' / 'golden_review_queue.csv'
MINIMUM_REVIEWED = 80

report = audit_readiness(
    db_path=str(DB_PATH),
    kb_id=KB_ID,
    review_queue_path=str(REVIEW_QUEUE),
    minimum_reviewed=MINIMUM_REVIEWED,
)
assert report['status'] in {'ready', 'blocked'}
report['status'], report['summary']

('blocked', {'fail': 4, 'warn': 3, 'pass': 3})

## Data

Only aggregate counts and boolean scope indicators are retained in notebook output. Source message bodies and identifiers remain in the local ignored database and are never rendered here.

In [2]:
{
    'knowledge_base': report['kb'],
    'knowledge_files': report['knowledge_files'],
    'golden_dataset': {
        key: report['golden_dataset'][key]
        for key in ('total', 'active', 'reviewed', 'smoke', 'source_backed', 'todo_answer')
    },
    'review_queue': {
        key: report['review_queue'][key]
        for key in ('available', 'rows', 'decision_counts', 'pii_redaction_counts')
    },
}

{'knowledge_base': {'key': 'default',
  'name': 'All kb',
  'status': 'active',
  'access_level': 'public',
  'tenant_scoped': False,
  'org_scoped': False,
  'kb_version': '649a17d4b95c'},
 'knowledge_files': {'total': 42,
  'ingested': 29,
  'reviewed': 1,
  'published': 1,
  'failed_or_stale': 40},
 'golden_dataset': {'total': 150,
  'active': 20,
  'reviewed': 0,
  'smoke': 19,
  'source_backed': 52,
  'todo_answer': 107},
 'review_queue': {'available': True,
  'rows': 66,
  'decision_counts': {'pending': 66},
  'pii_redaction_counts': {'phone_redacted': 5, 'email_redacted': 4}}}

## Results

In [3]:
[
    {
        'check': finding['key'],
        'status': finding['status'],
        'message': finding['message'],
        'evidence': finding['evidence'],
    }
    for finding in report['findings']
]

[{'check': 'kb_active',
  'status': 'pass',
  'message': 'KB status is active',
  'evidence': {}},
 {'check': 'kb_access_scope',
  'status': 'fail',
  'message': 'Pilot KB must not be public',
  'evidence': {'access_level': 'public'}},
 {'check': 'kb_tenant_scope',
  'status': 'fail',
  'message': 'Pilot KB needs tenant_id or org_id scope',
  'evidence': {'tenant_scoped': False, 'org_scoped': False}},
 {'check': 'trusted_knowledge_sources',
  'status': 'pass',
  'message': 'At least one reviewed or published source is required',
  'evidence': {'reviewed': 1, 'published': 1}},
 {'check': 'knowledge_debt',
  'status': 'warn',
  'message': 'Failed or stale sources should be resolved before pilot',
  'evidence': {'failed_or_stale': 40}},
 {'check': 'reviewed_golden_coverage',
  'status': 'fail',
  'message': 'At least 80 owner-reviewed golden rows are required',
  'evidence': {'reviewed': 0, 'active': 20}},
 {'check': 'golden_bucket_coverage',
  'status': 'fail',
  'message': 'Golden data 

In [4]:
{
    'support_operations': report['support_operations'],
    'pii_shape_counts_by_source': report['source_pii_shape_counts'],
}
# These are shape counts only; no matched values are exposed.

{'support_operations': {'tickets': 10,
  'resolved_or_closed_tickets': 1,
  'inbound_emails': 615,
  'replied_emails': 1,
  'workflow_runs': 0},
 'pii_shape_counts_by_source': {'chat_logs': {'rows': 909,
   'email_shape_rows': 1,
   'phone_shape_rows': 133,
   'secret_shape_rows': 0},
  'support_tickets': {'rows': 10,
   'email_shape_rows': 7,
   'phone_shape_rows': 1,
   'secret_shape_rows': 0},
  'support_email_messages': {'rows': 616,
   'email_shape_rows': 616,
   'phone_shape_rows': 376,
   'secret_shape_rows': 50}}}

## Takeaways

1. Use a dedicated internal, tenant- or org-scoped knowledge base; the current mixed public KB must not be promoted directly.
2. Assign named business, knowledge, approval, and technical owners before activating any candidate.
3. Review the redacted queue and approve only evidence-backed or explicitly negative cases.
4. Re-run this notebook and the benchmark promotion gate after each review batch.
5. Begin a restricted pilot only when all blocking checks pass; keep write actions behind approval throughout Phase 1.